In [16]:
import pandas as pd
import geobr
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

from soja_clima.agregacao import agregar_diario_para_estacao_safra, agregar_espacial_para_microrregiao
from soja_clima.sidra import preparar_soja_microrregiao
from soja_clima.merge import montar_base_final

In [23]:
micro = geobr.read_micro_region(
    year=2020,
    simplified=True
)

micro['chave_join'] = (
    micro['name_micro'].str.strip().str.casefold() + '_' + micro['abbrev_state'].str.casefold()
)
micro.head()

,code_micro,name_micro,code_state,abbrev_state,name_state,code_region,name_region,year,geometry,chave_join
0,11001.0,Porto Velho,11.0,RO,Rondônia,1.0,Norte,2020.0,"POLYGON ((-63.10218 -9.47892, -63.07806 -9.508...",porto velho_ro
1,11002.0,Guajará-Mirim,11.0,RO,Rondônia,1.0,Norte,2020.0,"POLYGON ((-64.60819 -12.21637, -64.58383 -12.2...",guajará-mirim_ro
2,11003.0,Ariquemes,11.0,RO,Rondônia,1.0,Norte,2020.0,"POLYGON ((-63.07806 -9.50802, -63.10218 -9.478...",ariquemes_ro
3,11004.0,Ji-Paraná,11.0,RO,Rondônia,1.0,Norte,2020.0,"POLYGON ((-62.66998 -10.28828, -62.67077 -10.2...",ji-paraná_ro
4,11005.0,Alvorada d'Oeste,11.0,RO,Rondônia,1.0,Norte,2020.0,"POLYGON ((-63.60307 -11.60143, -63.61343 -11.6...",alvorada d'oeste_ro


In [24]:
path = "../data/raw/dadosSidra2005_2024.csv"

# Acha automaticamente onde a tabela termina, procurando a linha "Fonte:"
# que o SIDRA/IBGE sempre coloca logo após os dados
with open(path, encoding='utf-8-sig') as f:
    linhas = f.readlines()

linha_fonte = next(i for i, l in enumerate(linhas) if l.strip().startswith('"Fonte:'))
n_dados = linha_fonte - 3   # linhas 0-1 = título/cabeçalho truncado, linha 2 = header real

df = pd.read_csv(
    path,
    sep=';',
    encoding='utf-8-sig',   # o arquivo tem BOM UTF-8
    skiprows=2,             # pula título + cabeçalho truncado
    nrows=n_dados,          # lê só até a última linha antes de "Fonte:"
    quotechar='"',
) 

extraido = df['Microrregião Geográfica'].str.extract(r'^(.*?)\s*\(([A-Z]{2})\)$')
df['nome_micro'] = extraido[0].str.strip()
df['uf'] = extraido[1]
df['chave_join'] = df['nome_micro'].str.casefold() + '_' + df['uf'].str.casefold()

df.head()

,Microrregião Geográfica,Ano,#,Produto das lavouras temporárias,Área plantada (Hectares),Área plantada - percentual do total geral,Área colhida (Hectares),Área colhida - percentual do total geral,Quantidade produzida (Toneladas),Rendimento médio da produção (Quilogramas por Hectare),nome_micro,uf,chave_join
0,Porto Velho (RO),2005,1,Soja (em grão),455,"1,27",455,"1,27",1365,3000,Porto Velho,RO,porto velho_ro
1,Porto Velho (RO),2006,1,Soja (em grão),710,"1,97",710,"1,97",1706,2402,Porto Velho,RO,porto velho_ro
2,Porto Velho (RO),2007,1,Soja (em grão),710,"1,94",680,"1,89",1768,2600,Porto Velho,RO,porto velho_ro
3,Porto Velho (RO),2008,1,Soja (em grão),380,"1,04",380,"1,07",984,2589,Porto Velho,RO,porto velho_ro
4,Porto Velho (RO),2009,1,Soja (em grão),300,"0,81",300,"0,81",720,2400,Porto Velho,RO,porto velho_ro


In [25]:
producao_soja_com_micro = micro.merge(
    df,
    on='chave_join',
    how='outer'
).rename(columns={'Ano': 'ano'})

producao_soja_com_micro[producao_soja_com_micro['code_micro'].isna()]

,code_micro,name_micro,code_state,abbrev_state,name_state,code_region,name_region,year,geometry,chave_join,...,#,Produto das lavouras temporárias,Área plantada (Hectares),Área plantada - percentual do total geral,Área colhida (Hectares),Área colhida - percentual do total geral,Quantidade produzida (Toneladas),Rendimento médio da produção (Quilogramas por Hectare),nome_micro,uf


In [26]:
producao_soja_com_micro[producao_soja_com_micro['Microrregião Geográfica'].isna()]

,code_micro,name_micro,code_state,abbrev_state,name_state,code_region,name_region,year,geometry,chave_join,...,#,Produto das lavouras temporárias,Área plantada (Hectares),Área plantada - percentual do total geral,Área colhida (Hectares),Área colhida - percentual do total geral,Quantidade produzida (Toneladas),Rendimento médio da produção (Quilogramas por Hectare),nome_micro,uf


In [27]:
path = "../data/processed/clima_diario_agg.parquet"

diario = pd.read_parquet(path)
diario.head()

,codigo_estacao,uf,estacao,latitude,longitude,data,prec_total,temp_media,temp_max,temp_min,umid_media,radiacao_total,vento_medio,n_horas_validas,amplitude_termica,dia_seco,gdd
0,A001,DF,BRASILIA,-15.789444,-47.925833,2005-01-01,0.4,21.204167,26.0,19.1,81.541667,8642.0,1.533333,24,6.9,True,12.55
1,A001,DF,BRASILIA,-15.789444,-47.925833,2005-01-02,8.8,20.800000,26.9,18.3,83.208333,3868.0,1.979167,24,8.6,False,12.60
2,A001,DF,BRASILIA,-15.789444,-47.925833,2005-01-03,10.8,19.833333,24.1,17.8,85.875000,6146.0,3.091667,24,6.3,False,10.95
3,A001,DF,BRASILIA,-15.789444,-47.925833,2005-01-04,16.2,20.595833,27.5,17.4,84.333333,6850.0,2.154167,24,10.1,False,12.45
4,A001,DF,BRASILIA,-15.789444,-47.925833,2005-01-05,15.8,20.804167,25.7,17.5,82.583333,2697.0,2.116667,24,8.2,False,11.60


In [28]:
from shapely.geometry import Point

microrregioes = geobr.read_micro_region(year=2020)  # ajuste o ano se necessário

print(microrregioes.crs)          # confira o CRS de referência
print(microrregioes.columns.tolist())

EPSG:4674
['code_micro', 'name_micro', 'code_state', 'abbrev_state', 'name_state', 'code_region', 'name_region', 'year', 'geometry']


In [29]:
# ============================================================
# CÉLULA 3 — Montar o GeoDataFrame das estações
# ============================================================
# Parte da tabela de estações únicas que você já extrai no pipeline
# climático (codigo_estacao, uf, estacao, latitude, longitude).
# Se ainda não tiver essa tabela separada, extraia do diario_unido:
estacoes = diario[['codigo_estacao', 'estacao', 'uf', 'latitude', 'longitude']].drop_duplicates()

# Uma mesma estação pode ter coordenadas levemente diferentes entre
# arquivos de anos diferentes (deriva de metadado no cabeçalho do
# INMET). Isso gera linhas "duplicadas" que na verdade são a mesma
# estação física -- se não resolver aqui, ela acaba contando duas vezes
# na agregação climática mais adiante, além de poder quebrar joins que
# dependem de codigo_estacao ser único.
duplicadas = estacoes[estacoes.duplicated('codigo_estacao', keep=False)]
if len(duplicadas) > 0:
    print(f"{duplicadas['codigo_estacao'].nunique()} estação(ões) com coordenadas "
          "divergentes entre arquivos -- usando a mediana das coordenadas:")
    print(duplicadas.sort_values('codigo_estacao'))

estacoes = (estacoes.groupby('codigo_estacao', as_index=False)
            .agg(estacao=('estacao', 'first'), uf=('uf', 'first'),
                 latitude=('latitude', 'median'), longitude=('longitude', 'median')))

geometria_estacoes = [Point(lon, lat) for lon, lat in zip(estacoes['longitude'], estacoes['latitude'])]
# Point(x, y) = Point(longitude, latitude) -- shapely usa (x, y), não (lat, lon).
# Inverter essa ordem é um erro clássico e silencioso (o ponto cai no
# hemisfério/quadrante errado sem lançar erro nenhum).

estacoes_geo = gpd.GeoDataFrame(estacoes, geometry=geometria_estacoes, crs="EPSG:4326")
# EPSG:4326 = WGS84, o sistema padrão de lat/lon (graus decimais).
# É o mesmo sistema que o INMET usa para as coordenadas do cabeçalho.

# Garantir que as estações estão no MESMO CRS da malha de microrregiões
# antes de comparar geometrias -- CRS diferente dá resultado errado sem
# lançar erro.
estacoes_geo = estacoes_geo.to_crs(microrregioes.crs)


# ============================================================
# CÉLULA 4 — Join espacial (point-in-polygon)
# ============================================================
estacoes_com_micro = gpd.sjoin(
    estacoes_geo,
    microrregioes[['code_micro', 'name_micro', 'geometry']],
    how='left',
    predicate='within',   # "all versão recente do geopandas usa predicate; versões antigas usam op"
)

print(f"{estacoes_com_micro['code_micro'].isna().sum()} estação(ões) sem microrregião correspondente.")
print(estacoes_com_micro[estacoes_com_micro['code_micro'].isna()][['codigo_estacao', 'estacao', 'uf', 'latitude', 'longitude']])

548 estação(ões) com coordenadas divergentes entre arquivos -- usando a mediana das coordenadas:
        codigo_estacao              estacao  uf   latitude  longitude
0                 A001             BRASILIA  DF -15.789444 -47.925833
1827107           A001             BRASILIA  DF -15.789343 -47.925756
348               A002              GOIANIA  GO -16.642778 -49.220000
1828911           A002              GOIANIA  GO -16.642841 -49.220222
2429210           A002              GOIANIA  GO -16.642778 -49.220278
...                ...                  ...  ..        ...        ...
1660367           S715   RIBAS DO RIO PARDO  MS -20.466667 -53.763056
1660652           S716  SANTA RITA DO PARDO  MS -21.305833 -52.820278
1849115           S716  SANTA RITA DO PARDO  MS -21.305889 -52.820375
1849328           S717             SELVIRIA  MS -20.351444 -51.430222
1660904           S717             SELVIRIA  MS -20.351389 -51.430278

[1423 rows x 5 columns]
2 estação(ões) sem microrregião corres

In [30]:
sem_correspondencia = estacoes_com_micro[estacoes_com_micro['code_micro'].isna()].copy()

# Distância acima da qual a "estação/microrregião mais próxima" não tem
# mais sentido físico como representante (ex: uma estação da Antártica
# ou de um arquipélago oceânico distante não deveria herdar o clima de
# uma microrregião continental a milhares de km).
DISTANCIA_MAXIMA_KM = 150

if len(sem_correspondencia) > 0:
    microrregioes_metros = microrregioes.to_crs(epsg=5880)
    sem_correspondencia_metros = sem_correspondencia[['codigo_estacao', 'geometry']].to_crs(epsg=5880)

    mais_proximo = gpd.sjoin_nearest(
        sem_correspondencia_metros,
        microrregioes_metros[['code_micro', 'name_micro', 'geometry']],
        how='left',
        distance_col='distancia_m',
    )
    mais_proximo['distancia_km'] = mais_proximo['distancia_m'] / 1000

    print("Distância (km) até a microrregião mais próxima, para os casos sem correspondência direta:")
    print(mais_proximo[['codigo_estacao', 'name_micro', 'distancia_km']].round(1)
          .sort_values('distancia_km', ascending=False))

    # Só aceita o fallback dentro do limiar; acima disso, a estação fica
    # de fora do mapa (não faz sentido forçar uma atribuição a milhares
    # de km de distância -- ex: estação da Antártica, arquipélagos).
    dentro_do_limiar = mais_proximo[mais_proximo['distancia_km'] <= DISTANCIA_MAXIMA_KM]
    excluidas = mais_proximo[mais_proximo['distancia_km'] > DISTANCIA_MAXIMA_KM]

    if len(excluidas) > 0:
        print(f"\n{len(excluidas)} estação(ões) excluída(s) por estarem a mais de "
              f"{DISTANCIA_MAXIMA_KM}km de qualquer microrregião (provavelmente fora do "
              "território continental, ex: Antártica, arquipélagos oceânicos):")
        print(excluidas[['codigo_estacao', 'name_micro', 'distancia_km']].round(1))

    # merge (não .loc por índice) -- funciona mesmo se codigo_estacao tiver
    # duplicatas na tabela de estações, sem quebrar por índice ambíguo.
    # Isso também é o sinal para investigar a duplicata na origem: a
    # mesma estação com coordenadas levemente diferentes entre arquivos
    # de anos diferentes (deriva de metadado), o que merece dedupe
    # anterior na construção de `estacoes` se possível.
    substituicoes = (dentro_do_limiar[['codigo_estacao', 'code_micro', 'name_micro']]
                      .drop_duplicates('codigo_estacao')
                      .rename(columns={'code_micro': 'code_micro_sub', 'name_micro': 'name_micro_sub'}))

    estacoes_com_micro = estacoes_com_micro.merge(substituicoes, on='codigo_estacao', how='left')
    estacoes_com_micro['code_micro'] = estacoes_com_micro['code_micro'].fillna(estacoes_com_micro['code_micro_sub'])
    estacoes_com_micro['name_micro'] = estacoes_com_micro['name_micro'].fillna(estacoes_com_micro['name_micro_sub'])
    estacoes_com_micro = estacoes_com_micro.drop(columns=['code_micro_sub', 'name_micro_sub'])

    print(f"\n{estacoes_com_micro['code_micro'].isna().sum()} estação(ões) ainda sem "
          "correspondência após fallback (excluídas do mapa final).")

Distância (km) até a microrregião mais próxima, para os casos sem correspondência direta:
    codigo_estacao           name_micro  distancia_km
580           C891      Litoral Lagunar        5670.8
133           A302  Fernando de Noronha         657.9

2 estação(ões) excluída(s) por estarem a mais de 150km de qualquer microrregião (provavelmente fora do território continental, ex: Antártica, arquipélagos oceânicos):
    codigo_estacao           name_micro  distancia_km
133           A302  Fernando de Noronha         657.9
580           C891      Litoral Lagunar        5670.8

2 estação(ões) ainda sem correspondência após fallback (excluídas do mapa final).


In [31]:
mapa_estacao_micro = estacoes_com_micro[['codigo_estacao', 'estacao', 'uf', 'code_micro', 'name_micro']]
mapa_estacao_micro.head()

,codigo_estacao,estacao,uf,code_micro,name_micro
0,A001,BRASILIA,DF,53001.0,Brasília
1,A002,GOIANIA,GO,52010.0,Goiânia
2,A003,MORRINHOS,GO,52015.0,Meia Ponte
3,A005,PORANGATU,GO,52004.0,Porangatu
4,A009,PALMAS,TO,17006.0,Porto Nacional


In [32]:
# A partir daqui, junte esse mapa de volta no diario (ou no agregado por
# safra) e agregue por code_micro em vez de uf:
#
# diario_com_micro = diario.merge(mapa_estacao_micro[['codigo_estacao', 'code_micro']],
#                                  on='codigo_estacao', how='left')
# clima_micro_safra = diario_com_micro.groupby(['code_micro', 'safra']).agg(...)


# ============================================================
# CÉLULA 7 — Microrregiões que não receberam NENHUMA estação
# ============================================================
# O sjoin só rotula estações que caem dentro de algum polígono. Uma
# microrregião sem nenhuma estação dentro dela simplesmente não aparece
# depois do groupby -- não vira NaN, ela desaparece silenciosamente.
# Aqui invertemos a lógica: para essas microrregiões vazias, perguntamos
# "qual é a estação mais próxima do MEU centro?", em vez de "qual
# estação cai dentro de mim" (que já deu zero).

microrregioes_com_estacao = set(estacoes_com_micro['code_micro'].dropna())
todas_microrregioes = set(microrregioes['code_micro'])
microrregioes_vazias = todas_microrregioes - microrregioes_com_estacao

print(f"{len(microrregioes_vazias)} de {len(todas_microrregioes)} microrregiões "
      f"não têm nenhuma estação dentro delas.")

162 de 558 microrregiões não têm nenhuma estação dentro delas.


In [33]:
if microrregioes_vazias:
    micro_vazias_gdf = microrregioes[microrregioes['code_micro'].isin(microrregioes_vazias)].copy()

    # Centróide de cada microrregião vazia, em CRS métrico para a
    # distância fazer sentido fisicamente (não em graus de lat/lon)
    micro_vazias_metros = micro_vazias_gdf.to_crs(epsg=5880)
    micro_vazias_metros['geometry'] = micro_vazias_metros.centroid

    estacoes_metros = estacoes_geo.to_crs(epsg=5880)

    estacao_substituta = gpd.sjoin_nearest(
        micro_vazias_metros[['code_micro', 'name_micro', 'geometry']],
        estacoes_metros[['codigo_estacao', 'estacao', 'geometry']],
        how='left',
        distance_col='distancia_m',
    )

    print("\nDistância (km) até a estação substituta, para as microrregiões sem estação própria:")
    print(estacao_substituta.assign(distancia_km=lambda d: (d['distancia_m']/1000).round(0))
          [['code_micro', 'name_micro', 'codigo_estacao', 'distancia_km']]
          .sort_values('distancia_km', ascending=False).head(15))

    # Mesmo limiar da Célula 5: acima disso, não faz sentido físico
    # emprestar o clima de uma estação tão distante.
    estacao_substituta['distancia_km'] = estacao_substituta['distancia_m'] / 1000
    longe_demais = estacao_substituta[estacao_substituta['distancia_km'] > DISTANCIA_MAXIMA_KM]
    if len(longe_demais) > 0:
        print(f"\n{len(longe_demais)} microrregião(ões) sem substituta dentro de "
              f"{DISTANCIA_MAXIMA_KM}km -- ficarão sem clima (não entrarão no merge final):")
        print(longe_demais[['code_micro', 'name_micro', 'distancia_km']].round(1))
    estacao_substituta = estacao_substituta[estacao_substituta['distancia_km'] <= DISTANCIA_MAXIMA_KM]

    # Anexa essas microrregiões "órfãs" na tabela final, apontando para
    # a estação substituta -- assim elas também entram na agregação
    orfas = estacao_substituta[['code_micro', 'name_micro', 'codigo_estacao']].copy()
    orfas['origem'] = 'substituta_por_centroide'
    mapa_estacao_micro['origem'] = 'dentro_do_poligono'
    mapa_estacao_micro_completo = pd.concat([mapa_estacao_micro, orfas], ignore_index=True)
else:
    mapa_estacao_micro_completo = mapa_estacao_micro.assign(origem='dentro_do_poligono')

print(f"\nCobertura final: {mapa_estacao_micro_completo['code_micro'].nunique()} "
      f"de {len(todas_microrregioes)} microrregiões com alguma estação (própria ou substituta).")


Distância (km) até a estação substituta, para as microrregiões sem estação própria:
     code_micro                name_micro codigo_estacao  distancia_km
193     26019.0       Fernando de Noronha           A344         373.0
14      13002.0                    Japurá           A134         205.0
28      14003.0                 Caracaraí           A135         190.0
55      16004.0                   Mazagão           A244         162.0
4       11005.0          Alvorada d'Oeste           A939         159.0
7       11008.0         Colorado do Oeste           A938         124.0
53      16002.0                     Amapá           A251         111.0
3       11004.0                 Ji-Paraná           A940         102.0
78      21015.0                      Codó           A237          88.0
69      21006.0           Itapecuru Mirim           A206          88.0
527     51011.0            Médio Araguaia           A908          84.0
84      21021.0  Chapadas das Mangabeiras           A204       

C:\Users\okubo\AppData\Local\Temp\ipykernel_16604\4011633432.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mapa_estacao_micro['origem'] = 'dentro_do_poligono'


In [34]:
# Dias esperados em cada fase (derivado dos limiares já usados:
# plantio 0-39, vegetativo 40-89, floracao_enchimento 90-129,
# maturacao 130-240).
DIAS_ESPERADOS_FASE = {
    'plantio': 40,
    'vegetativo': 50,
    'floracao_enchimento': 40,
    'maturacao': 111,
}

LIMIAR_COMPLETUDE = 0.80  # abaixo disso, o valor agregado vira NaN

In [41]:
diario_com_micro = diario.merge(
    mapa_estacao_micro_completo[['codigo_estacao', 'code_micro']],
    on='codigo_estacao', how='left'
)

# producao_soja_com_micro já é o resultado do seu join SIDRA + geobr
# (colunas: code_micro, name_micro, ..., Ano, Área plantada (Hectares), ...)

base_estacao_safra = agregar_diario_para_estacao_safra(diario_com_micro, usar_fases=False)
clima_micro_safra = agregar_espacial_para_microrregiao(base_estacao_safra, fases=['safra_completa'])
producao_micro_ano = preparar_soja_microrregiao(producao_soja_com_micro)
producao_micro_ano = producao_micro_ano[producao_micro_ano['producao_ton'].fillna(0) > 0].copy()
base_final = montar_base_final(clima_micro_safra, producao_micro_ano)

print("Shape final:", base_final.shape)


RAIZ_PROJETO = Path.cwd().parent
PASTA_PROCESSADA = RAIZ_PROJETO / "data" / "processed"
caminho_saida = PASTA_PROCESSADA / "base_inmet_sidra.parquet"

base_final.to_parquet(caminho_saida, index=False)

OK: cada (code_micro, ano) aparece exatamente uma vez.
OK: rendimento recalculado bate com o do SIDRA em todas as linhas.
Shape final: (4413, 19)


In [42]:
print(base_final.shape)
print(base_final.groupby(['code_micro', 'safra']).size().value_counts())  # deveria ser só "1"

(4413, 19)
1    4413
Name: count, dtype: int64


In [43]:
total_micro_com_soja = producao_micro_ano['code_micro'].nunique()
total_micro_na_base_final = base_final['code_micro'].nunique()
print(f"{total_micro_na_base_final} de {total_micro_com_soja} microrregiões com soja restaram na base final")

329 de 334 microrregiões com soja restaram na base final


In [38]:
pd.read_parquet("../data/processed/base_inmet_sidra.parquet")

,code_micro,name_micro,abbrev_state,ano,area_plantada_ha,area_colhida_ha,producao_ton,rendimento_kg_ha,safra,prec_total_safra_completa,dias_secos_safra_completa,temp_media_safra_completa,gdd_acumulado_safra_completa,amplitude_media_safra_completa,radiacao_media_safra_completa,temp_max_abs_safra_completa,temp_min_abs_safra_completa,completude_safra_completa,n_estacoes
0,35035.0,Adamantina,SP,2008,850.0,850.0,1680.0,1976.0,2008,756.2,163.0,25.175696,3830.80,11.918333,19977.140833,39.7,12.7,0.995851,1.0
1,35035.0,Adamantina,SP,2009,700.0,700.0,1480.0,2114.0,2009,1047.8,167.0,24.824152,3762.05,12.426971,20347.907054,39.3,7.0,1.000000,1.0
2,35035.0,Adamantina,SP,2010,500.0,500.0,1000.0,2000.0,2010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.157676,1.0
3,35035.0,Adamantina,SP,2011,500.0,500.0,1000.0,2000.0,2011,879.6,178.0,24.599996,3743.50,11.665560,19276.402490,37.5,11.9,1.000000,1.0
4,35035.0,Adamantina,SP,2012,950.0,950.0,1772.0,1865.0,2012,1114.8,175.0,24.954391,3791.45,13.100000,21552.558506,38.3,8.5,1.000000,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8463,15001.0,Óbidos,PA,2019,NaN,NaN,NaN,NaN,2019,1635.0,129.0,26.359302,4243.60,9.848963,16711.319502,38.3,20.4,1.000000,1.0
8464,15001.0,Óbidos,PA,2020,NaN,NaN,NaN,NaN,2020,1559.0,132.0,26.185650,4226.15,9.968880,16929.317427,37.6,20.8,1.000000,1.0
8465,15001.0,Óbidos,PA,2021,NaN,NaN,NaN,NaN,2021,1333.0,127.0,25.853899,3731.20,10.105069,16547.451152,37.1,20.2,0.900415,1.0
8466,15001.0,Óbidos,PA,2023,NaN,NaN,NaN,NaN,2023,1888.0,121.0,25.484704,4015.65,9.147303,16085.052697,37.1,19.6,1.000000,1.0


In [45]:
microrregioes_produtoras = producao_micro_ano['code_micro'].unique()

produtoras_fora_da_base = set(microrregioes_produtoras) - set(base_final['code_micro'])
print(len(produtoras_fora_da_base), "microrregiões produtoras ficaram de fora\n")

detalhe = producao_micro_ano[producao_micro_ano['code_micro'].isin(produtoras_fora_da_base)]
print(detalhe[['code_micro', 'name_micro', 'abbrev_state', 'ano', 'producao_ton']]
      .sort_values('producao_ton', ascending=False))

sem_clima_algum = set(microrregioes_produtoras) - set(mapa_estacao_micro_completo['code_micro'])
print(f"\n{len(sem_clima_algum)} de {len(produtoras_fora_da_base)} ficaram de fora por falta de clima")

outro_motivo = produtoras_fora_da_base - sem_clima_algum
print(f"{len(outro_motivo)} ficaram de fora por outro motivo (ex: completude baixa na safra)")

5 microrregiões produtoras ficaram de fora

      code_micro          name_micro abbrev_state   ano  producao_ton
559      11005.0    Alvorada d'Oeste           RO  2024      136438.0
558      11005.0    Alvorada d'Oeste           RO  2023      127102.0
557      11005.0    Alvorada d'Oeste           RO  2022      100181.0
556      11005.0    Alvorada d'Oeste           RO  2021       59991.0
554      11005.0    Alvorada d'Oeste           RO  2019       45891.0
553      11005.0    Alvorada d'Oeste           RO  2018       43494.0
555      11005.0    Alvorada d'Oeste           RO  2020       42312.0
552      11005.0    Alvorada d'Oeste           RO  2017       39036.0
2559     14003.0           Caracaraí           RR  2024       35285.0
550      11005.0    Alvorada d'Oeste           RO  2015       35076.0
2558     14003.0           Caracaraí           RR  2023       31787.0
551      11005.0    Alvorada d'Oeste           RO  2016       27576.0
2557     14003.0           Caracaraí          

In [46]:
tres_com_estacao_mas_sem_dado = outro_motivo  # já calculado antes
mapa_estacao_micro_completo[mapa_estacao_micro_completo['code_micro'].isin(tres_com_estacao_mas_sem_dado)]

codigos_suspeitos = mapa_estacao_micro_completo[
    mapa_estacao_micro_completo['code_micro'].isin(tres_com_estacao_mas_sem_dado)
]['codigo_estacao']

base_estacao_safra[base_estacao_safra['codigo_estacao'].isin(codigos_suspeitos)][
    ['codigo_estacao', 'code_micro', 'safra', 'completude_safra_completa']
].sort_values('safra')

,codigo_estacao,code_micro,safra,completude_safra_completa
963,A121,13009.0,2008,0.024896
964,A121,13009.0,2009,1.000000
965,A121,13009.0,2010,0.991701
966,A121,13009.0,2011,1.000000
967,A121,13009.0,2012,0.892116
968,A121,13009.0,2013,0.979253
845,A109,13004.0,2013,0.887967
846,A109,13004.0,2014,1.000000
969,A121,13009.0,2014,1.000000
970,A121,13009.0,2015,1.000000


In [47]:
clima_micro_safra['code_micro'] = clima_micro_safra['code_micro'].astype(str)
producao_micro_ano['code_micro'] = producao_micro_ano['code_micro'].astype(str)

In [48]:
print(clima_micro_safra['code_micro'].dtype, producao_micro_ano['code_micro'].dtype)

codigos_teste = [13009, 13004, 14004]
print(clima_micro_safra[clima_micro_safra['code_micro'].isin(codigos_teste)])
print(producao_micro_ano[producao_micro_ano['code_micro'].isin(codigos_teste)][['code_micro']].drop_duplicates())

object object
Empty DataFrame
Columns: [code_micro, safra, prec_total_safra_completa, dias_secos_safra_completa, temp_media_safra_completa, gdd_acumulado_safra_completa, amplitude_media_safra_completa, radiacao_media_safra_completa, temp_max_abs_safra_completa, temp_min_abs_safra_completa, completude_safra_completa, n_estacoes]
Index: []
Empty DataFrame
Columns: [code_micro]
Index: []
